# Cutout → Structure Tensor → ODF Glyph

**Three-step pipeline:**
1. **Cutout** — download a 3-D ROI from a Neuroglancer link (`ac_data_tools`, optional) or load a local `.tif`
2. **Structure Tensor** — compute principal orientation vectors with `fiberorient.StructureTensor`
3. **ODF Glyph** — build a smoothed ODF (von Mises kernel) and render an anatomically-oriented glyph

> **Axis convention**: `cutout_from_NG` returns `(x, y, z)`. We immediately transpose to `(z, y, x)` for `fiberorient`.


---
## Setup

Install dependencies (run once, then restart the kernel):

```bash
pip install fiberorient pyvista tifffile pillow matplotlib numpy scipy
# Optional — only needed for Option A (cutout from Neuroglancer):
pip install git+https://github.com/AllenInstitute/ac_data_tools.git
```

Also install this package from the repo root:
```bash
pip install -e ..
```


In [ ]:
import json
import pathlib

import numpy as np
import tifffile as tiff
import matplotlib.pyplot as plt
from PIL import Image

import pyvista as pv
pv.set_jupyter_backend('static')   # off-screen rendering inside notebooks

from fiberorient import StructureTensor
from fiberorient.util import make_sphere

from st_odf import (
    odf_from_vectors_simple,
    generalized_fractional_anisotropy,
    apply_anatomical_rotation,
)

print('Imports OK')


---
## Parameters

Edit the cell below to configure the full pipeline.


In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
CONFIG_JSON = pathlib.Path('../datasets_config.json')
DATASET_ID  = 'MY_DATASET'   # ← must match a key in datasets_config.json

# ── Neuroglancer link (Option A only) ─────────────────────────────────────────
NG_LINK    = ''   # paste Neuroglancer share link here
LAYER_NAME = None   # None → auto-detect first image layer

# ── Cutout size & output ──────────────────────────────────────────────────────
CUTOUT_SIZE = (256, 256, 256)   # (z, y, x) voxels at the chosen MIP level
MIP_LEVEL   = '2'               # '0' = full resolution, '2' = 4× downsampled
OUT_DIR     = pathlib.Path('./cutouts')
POST_FIX    = 'pos1'            # appended to the output filename

# ── Structure Tensor ──────────────────────────────────────────────────────────
ST_D_SIGMA          = 1   # derivative sigma
ST_N_SIGMA          = 1   # neighbourhood sigma
INTENSITY_THRESHOLD = 0   # exclude voxels below this intensity (0 = no mask)

# ── ODF ───────────────────────────────────────────────────────────────────────
ODF_METHOD    = 'vonmises'   # 'vonmises' | 'kde' | 'hist'
ODF_KAPPA     = 20           # concentration (vonmises; higher = sharper)
SPHERE_POINTS = 6500         # tessellation density of the unit sphere
MAX_VECTORS   = 125_000      # subsample to this many vectors before ODF

# ── Glyph & screenshot ────────────────────────────────────────────────────────
GLYPH_SCALE    = 3.0
WINDOW_SIZE    = (900, 900)
CAMERA_POS     = [(0, 0, 10), (0, 0, 0), (0, -1, 0)]   # (pos, focal, view_up)
SCREENSHOT_PNG = OUT_DIR / f'odf_glyph_{DATASET_ID}.png'


---
## Load Anatomical Rotation from Config


In [ ]:
with open(CONFIG_JSON) as f:
    cfg = json.load(f)

entry = cfg.get(DATASET_ID)
if entry is None:
    raise ValueError(f"'{DATASET_ID}' not found in {CONFIG_JSON}. Available: {list(cfg)}")

ANATOMICAL_ROTATION = np.array(
    entry.get('anatomical_rotation', [[1, 0, 0], [0, 1, 0], [0, 0, 1]])
)
print(f'Dataset         : {DATASET_ID}')
print(f'Anatomical rot  :\n{ANATOMICAL_ROTATION}')


---
## Step 1: Cutout

**Option A** — download from a Neuroglancer zarr store via `cutout_from_NG` (requires `ac_data_tools`).  
**Option B** — load a local `.tif` file.

> **Axis note**: `cutout_from_NG` returns `(x, y, z)`. We transpose immediately to `(z, y, x)` for `fiberorient`.


In [ ]:
# ── Option A: cutout from Neuroglancer ───────────────────────────────────────
from ac_data_tools.cutout import cutout_from_NG

OUT_DIR.mkdir(parents=True, exist_ok=True)

cutout, metadata = cutout_from_NG(
    ng_link    = NG_LINK,
    layer_name = LAYER_NAME,
    size       = CUTOUT_SIZE,
    out_path   = str(OUT_DIR),
    post_fix   = POST_FIX,
    mip        = MIP_LEVEL,
    save_tif   = True,
)
# cutout_from_NG returns (x, y, z); transpose to (z, y, x) for fiberorient
img = cutout.transpose(2, 1, 0)
TIF_PATH = pathlib.Path(metadata['tif_path'])
print(f'Image shape (z, y, x): {img.shape}')


In [ ]:
# ── Option B: local tif ───────────────────────────────────────────────────────
# Uncomment and set the path to your tif file.
# tifffile reads tifs saved by cutout_from_NG as (z, y, x) automatically.

# TIF_PATH = pathlib.Path('/path/to/your/cutout.tif')
# img = tiff.imread(str(TIF_PATH))
# print(f'Image shape (z, y, x): {img.shape}')


In [ ]:
print(f'Image shape (z, y, x): {img.shape}  dtype: {img.dtype}')
print(f'Intensity range: [{img.min()}, {img.max()}]')

mz, my, mx = [s // 2 for s in img.shape]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, proj, label in zip(
    axes,
    [img[mz, :, :], img[:, my, :].T, img[:, :, mx].T],
    [f'Mid-Z={mz} (YX plane)', f'Mid-Y={my} (XZ plane)', f'Mid-X={mx} (ZY plane)'],
):
    ax.imshow(proj, cmap='gray')
    ax.set_title(label)
    ax.axis('off')

plt.suptitle(f'Cutout middle planes — {DATASET_ID}', fontsize=13)
plt.tight_layout()
plt.show()


---
## Step 2: Structure Tensor

`fiberorient.StructureTensor` computes the gradient structure tensor and returns
the principal eigenvector (local fibre orientation) as a `(Z, Y, X, 3)` array.


In [ ]:
st = StructureTensor(d_sigma=ST_D_SIGMA, n_sigma=ST_N_SIGMA)
st.fit(img)
vectors = st.vectors   # shape: (Z, Y, X, 3)
print(f'Vectors shape: {vectors.shape}')


In [ ]:
# Direction-Encoded Colour (DEC) image
# Rotate vectors to anatomical space, then map |x|→R, |y|→G, |z|→B
vectors_anat = vectors @ ANATOMICAL_ROTATION.T   # (Z, Y, X, 3)

# Compute DEC manually — img_to_dec mutates its input in-place, so we avoid it
v = np.abs(vectors_anat)
v = v / v.max()
scalar = (img.astype(np.float32) - img.min()) / (img.max() - img.min() + 1e-8) * 255.0
dec_img = (scalar[..., None] * v).astype(np.uint8)   # (Z, Y, X, 3)

mz, my, mx = [s // 2 for s in img.shape]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, proj, label in zip(
    axes,
    [dec_img[mz, :, :], dec_img[:, my, :], dec_img[:, :, mx]],
    [f'DEC Mid-Z={mz}', f'DEC Mid-Y={my}', f'DEC Mid-X={mx}'],
):
    ax.imshow(proj)
    ax.set_title(label)
    ax.axis('off')
plt.suptitle('DEC middle planes  R=dim-0  G=dim-1  B=dim-2', fontsize=13)
plt.tight_layout()
plt.show()


---
## Step 3: ODF Glyph (Smoothed Histogram — von Mises)

Each foreground voxel contributes its principal orientation vector to a directional
histogram on the unit sphere. The von Mises kernel gives a smooth, continuous ODF.

The glyph is rendered off-screen with **PyVista** using the anatomical rotation from
the dataset config, then saved as a PNG.

> **Antipodal symmetry**: eigenvectors have arbitrary sign, so we include both **v** and **−v**
> before computing the ODF to ensure both hemispheres are populated.


In [ ]:
sphere = make_sphere(SPHERE_POINTS)

# Foreground mask
mask = (img > INTENSITY_THRESHOLD) if INTENSITY_THRESHOLD > 0 else np.ones(img.shape, dtype=bool)
vectors_fg = vectors[mask]
print(f'Foreground voxels : {vectors_fg.shape[0]:,}')

# Subsample for speed — statistics converge well before N=500K
if len(vectors_fg) > MAX_VECTORS:
    rng = np.random.default_rng(42)
    idx = rng.choice(len(vectors_fg), size=MAX_VECTORS, replace=False)
    vectors_odf = vectors_fg[idx]
    print(f'Subsampled to     : {MAX_VECTORS:,} vectors for ODF')
else:
    vectors_odf = vectors_fg

# Enforce antipodal symmetry: include both v and -v
vectors_odf = np.concatenate([vectors_odf, -vectors_odf], axis=0)

odf_on_sphere = odf_from_vectors_simple(
    vectors_odf, sphere.vertices,
    method=ODF_METHOD, kappa=ODF_KAPPA, normalize=True,
)

gfa = generalized_fractional_anisotropy(odf_on_sphere)
print(f'GFA : {gfa:.4f}')


In [ ]:
# ── 1. Build glyph mesh in anatomical orientation ────────────────────────────────────────────
rot_verts = apply_anatomical_rotation(sphere.vertices, ANATOMICAL_ROTATION)

values_clipped = np.clip(odf_on_sphere, 0, None)
vmax          = values_clipped.max()
values_norm   = values_clipped / vmax if vmax > 0 else values_clipped

verts = rot_verts * (values_norm * GLYPH_SCALE)[:, None]

# Direction-encoded RGB colours (derived from rotated sphere vertices)
abs_cv = np.abs(rot_verts)
max_cv = abs_cv.max(axis=1, keepdims=True)
max_cv[max_cv == 0] = 1
rgb = abs_cv / max_cv

# Build PyVista PolyData
if sphere.faces.ndim > 1:
    faces_padded = np.column_stack(
        [np.full(sphere.faces.shape[0], 3), sphere.faces]
    ).ravel()
else:
    faces_padded = sphere.faces

odf_mesh = pv.PolyData(verts, faces=faces_padded)
odf_mesh['colors'] = rgb

# ── 2. Render ────────────────────────────────────────────────────────────────────────────
plotter = pv.Plotter(window_size=WINDOW_SIZE, off_screen=True)
plotter.set_background('#0a0a0a')
plotter.add_mesh(
    odf_mesh, scalars='colors', rgb=True,
    opacity=0.85, smooth_shading=True,
)
plotter.add_text(
    f'{DATASET_ID} | {ODF_METHOD} κ={ODF_KAPPA} | GFA={gfa:.3f}',
    font_size=10, position='upper_edge', color='white',
)
plotter.add_axes()
plotter.camera_position = CAMERA_POS
plotter.camera.zoom(0.8)

SCREENSHOT_PNG.parent.mkdir(parents=True, exist_ok=True)
plotter.screenshot(str(SCREENSHOT_PNG))
print(f'Screenshot saved: {SCREENSHOT_PNG}')

# ── 3. Display inline ───────────────────────────────────────────────────────────────────
plt.figure(figsize=(8, 8))
plt.imshow(np.array(Image.open(SCREENSHOT_PNG)))
plt.axis('off')
plt.title(f'ODF Glyph — {DATASET_ID}', fontsize=13)
plt.tight_layout()
plt.show()


---
## Tips

| Goal | What to change |
|------|----------------|
| Different dataset | `DATASET_ID` + add entry to `datasets_config.json` |
| Different region | `NG_LINK` (paste a new Neuroglancer share link) |
| No Neuroglancer access | Use Option B in Step 1 to load a local `.tif` |
| Sharper / smoother ODF | `ODF_KAPPA` (higher = sharper for `vonmises`) |
| Different kernel | `ODF_METHOD = 'kde'` or `'hist'` |
| Change view angle | `CAMERA_POS` — e.g. `[(10,0,0),(0,0,0),(0,-1,0)]` for YZ view |
| Larger screenshot | `WINDOW_SIZE = (1200, 1200)` |
| Faster ODF | Reduce `MAX_VECTORS` or `SPHERE_POINTS` |
